# Shapelets: flexible source light

A single Sérsic can't capture a clumpy or irregular source. The `Shapelets` profile
expands the source surface brightness in a basis of `n_max` shapelet functions whose
**amplitudes are solved by least squares** — you sample only the scale `beta` and the
centre.

In [ ]:
%matplotlib inline
import numpy as np
import jax
from jax import numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
import matplotlib as mpl
from matplotlib import pyplot as plt
from corner import corner
from skimage.transform import downscale_local_mean
tfd = tfp.distributions

import gigalens
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.light.sersic import SersicEllipse  # (unused; kept for parity)
from gigalens.jax.profiles.light import shapelets as shapelets_mod
from gigalens.jax.profiles.light.shapelets import Shapelets
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.inference import ModellingSequence
print('jax', jax.__version__, '| devices', jax.devices())

## Build a realistic source from M101

To get a genuinely non-Sérsic source, this fits a shapelet basis to a real galaxy
image (M101) by least squares, giving amplitudes we can render a truth image from.

In [ ]:
n_max = 8
root = gigalens.__path__[0]
img = plt.imread(f'{root}/assets/M101.jpeg')
img = img[:, 131:-131]
img = jnp.sqrt(img[..., 1].astype(np.float32))
img -= np.median(img[:20, :20])
img *= 100
img = downscale_local_mean(img, (5, 5))
img = np.pad(img, [(100, 100), (100, 100)], mode='constant')

grid = jnp.linspace(-1, 1, len(img)).astype(np.float32)
xx, yy = jnp.meshgrid(grid, grid)
xx, yy = xx[jnp.newaxis, ..., jnp.newaxis], yy[jnp.newaxis, ..., jnp.newaxis]
shp = Shapelets(n_max=n_max, interpolate=False, use_lstsq=True)
components = jnp.squeeze(shp.light(xx, yy, beta=0.13, center_x=0, center_y=0))
X = components.reshape((shp.depth, -1))
sol, *_ = jnp.linalg.lstsq(X.T, img.flatten())
plt.imshow((sol @ X).reshape(img.shape)); plt.colorbar(); plt.title('Shapelet fit to M101'); plt.show()

## Model and truth

:::{admonition} lstsq vs explicit amplitudes
:class: tip
`use_lstsq=True` (fitting) solves the shapelet amplitudes each step — the source prior
has only `beta`, `center_x`, `center_y`. To *render* a known source, use
`use_lstsq=False` with explicit amplitude values (`Shapelets(n_max)._amp_names`).
:::

In [ ]:
amp_names = list(Shapelets(n_max)._amp_names)

# --- fitting model: EPL lens + lstsq shapelet source ---
epl = Component(EPL(), dict(
    theta_E=tfd.LogNormal(jnp.log(1.0), 0.25), gamma=tfd.TruncatedNormal(2, 0.25, 1, 3),
    e1=tfd.Normal(0, 0.1), e2=tfd.Normal(0, 0.1),
    center_x=tfd.Normal(0, 0.025), center_y=tfd.Normal(0, 0.025)))
source = Component(Shapelets(n_max=n_max, use_lstsq=True, interpolate=False), dict(
    beta=tfd.LogNormal(jnp.log(0.1), 0.15), center_x=tfd.Normal(0, 0.01), center_y=tfd.Normal(0, 0.01)))
model = LensModel([Plane(mass=[epl]), Plane(deflection_ratio=1.0, light=[source])])
names = list(model.z_param_names)
print('free parameters:', model.num_free_params, '| (shapelet amplitudes solved, not sampled)')

# --- truth: draw lens + source geometry from the prior, render source with amplitudes = sol ---
truth_unique = dict(model.prior.sample(seed=jax.random.PRNGKey(0)))
truth_unique['planes/1/light/0/beta'] = jnp.asarray(0.08, jnp.float32)
truth = np.array([float(truth_unique[n]) for n in names])

epl_truth = Component(EPL(), {k.split('/')[-1]: float(truth_unique[f'planes/0/mass/0/{k.split("/")[-1]}'])
                              for k in [n for n in names if n.startswith('planes/0/mass/0/')]})
src_truth = Component(Shapelets(n_max=n_max, use_lstsq=False, interpolate=False), dict(
    beta=0.08, center_x=float(truth_unique['planes/1/light/0/center_x']),
    center_y=float(truth_unique['planes/1/light/0/center_y']),
    **{name: float(sol[i]) for i, name in enumerate(amp_names)}))
truth_model = LensModel([Plane(mass=[epl_truth]), Plane(deflection_ratio=1.0, light=[src_truth])])

## Data: simulate and add noise

Render the truth model, add Gaussian noise, and wrap it in an `ImageData`. Least-squares
amplitudes mean `ProbModel(..., mode='lstsq')`.

In [ ]:
background_rms, exp_time = 0.1, 200
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=1)
clean = np.asarray(SceneSimulator(truth_model, sim_config).simulate(truth_model.to_params({})))
err_map = np.sqrt(background_rms**2 + np.clip(clean, 0, np.inf) / exp_time)
np.random.seed(1)
observed_img = clean + np.random.normal(scale=err_map)

ds = ImageData(observed_img, sim_config, background_rms=background_rms, exp_time=exp_time, sees='all')
prob = ProbModel(model, ds, mode='lstsq')
seq = ModellingSequence(prob)
plt.imshow(observed_img); plt.colorbar(); plt.title('Observed'); plt.show()

## Inference: MAP → SVI → HMC

This demo passes HMC tuning knobs (`init_eps`, `init_l`, `max_leapfrog_steps`) — useful
when the default step size struggles with a wide `n_max` basis.

In [ ]:
def to_constrained(z_rows):
    xb = model.bijector.forward(jnp.asarray(z_rows).reshape(-1, len(names)))
    return np.stack([np.asarray(xb[n]).reshape(-1) for n in names], axis=1)

opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
best_z, best_lp, _ = seq.MAP(opt, n_samples=100, num_steps=150, seed=0, output_type='best')
best_z = np.asarray(jax.device_get(best_z))
print('MAP log-post: %.4g' % float(best_lp))

In [ ]:
opt = optax.adabelief(1e-2, b1=0.95, b2=0.99)
qz, loss_hist = seq.SVI(best_z, opt, n_vi=500, num_steps=350)
plt.plot(np.asarray(loss_hist).reshape(-1)); plt.xlabel('step'); plt.ylabel('-ELBO'); plt.title('SVI loss'); plt.show()

In [ ]:
# Rebuild the SVI surrogate from host arrays: under JAX 0.10 the sharded qz trips a mesh
# (Manual vs Explicit) clash inside HMC's pmapped sampler.
qz = tfd.MultivariateNormalFullCovariance(
    loc=np.asarray(jax.device_get(qz.mean())),
    covariance_matrix=np.asarray(jax.device_get(qz.covariance())))
samples = seq.HMC(qz, n_hmc=50, init_eps=0.3, init_l=3, max_leapfrog_steps=300,
                  num_burnin_steps=250, num_results=750)
rhat = np.asarray(tfp.mcmc.potential_scale_reduction(samples, independent_chain_ndims=2))
ess = np.asarray(tfp.mcmc.effective_sample_size(samples, cross_chain_dims=[1, 2]))
print('max R-hat: %.3f | min ESS: %.0f' % (np.nanmax(rhat), np.nanmin(ess)))
n_params = samples.shape[-1]
post = to_constrained(np.asarray(samples).reshape(-1, n_params))

## Posterior and recovered source

Reconstruct the best-fit source with `prob.simulators[0].lstsq_simulate(...)` and compare
to truth.

In [ ]:
labels = [{'planes/0/mass/0/theta_E': r'$\theta_E$', 'planes/0/mass/0/gamma': r'$\gamma$',
           'planes/0/mass/0/e1': r'$e_1$', 'planes/0/mass/0/e2': r'$e_2$',
           'planes/0/mass/0/center_x': r'$x_{lens}$', 'planes/0/mass/0/center_y': r'$y_{lens}$',
           'planes/1/light/0/beta': r'$\beta$', 'planes/1/light/0/center_x': r'$x_{src}$',
           'planes/1/light/0/center_y': r'$y_{src}$'}[n] for n in names]
fig = corner(post, labels=labels, truths=truth, show_titles=True, title_fmt='.3f'); plt.show()

In [ ]:
best_unique = {n: jnp.asarray(np.mean(post[:, i])) for i, n in enumerate(names)}
params = model.to_params(best_unique)
sim = prob.simulators[0]
recon = np.asarray(sim.lstsq_simulate(params, ds.image, ds.error_map, ds.mask))
resid = (observed_img - recon) / np.asarray(ds.error_map)

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
im0 = ax[0].imshow(recon); ax[0].set_title('Recovered'); ax[0].axis('off'); plt.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(resid, cmap='bwr', vmin=-4, vmax=4); ax[1].set_title('Residual'); ax[1].axis('off'); plt.colorbar(im1, ax=ax[1])
ax[2].hist(resid.flatten(), range=(-4, 4), bins=30, density=True); ax[2].set_title('Residual histogram')
plt.tight_layout(); plt.show()